## Backtest resutls and visualization

Backtest resutls for strategies saved in the `results/run` directory. The notebook visualizes out of sample returns for the different strategies and summarizes the following metrics: 
- Expected shortfall at 80% confidence level (ES80)
- Annualized volatility (Vol)
- Sharpe ratio (SR)
- Soritno ratio (SOR)
- Maximum drawdown (MD)
- Expected shortfall ratio (ESR)
- Average asset turnover (AAT)
- Average currency turnover (ACT)

In [28]:
import os
import json

import pandas as pd
import matplotlib.pyplot as plt

RUNS_DIR = "runs"
OPTIMIZER_RUN_DIRS = [f"{RUNS_DIR}/{d}" for d in os.listdir(RUNS_DIR) if os.path.isdir(f"{RUNS_DIR}/{d}")]
RUN_TO_UNIVERSE_SIZE = {"1":10, "2":50, "3":100}

### Metrics summary table for all strategies backtested on increasing unvierses sizes

In [ ]:
rows = {}

for run_dir in OPTIMIZER_RUN_DIRS:
    run_nums = os.listdir(run_dir)
    for run_num in run_nums[:3]:  # only first 3 runs for each universe size
        json_path = f"{run_dir}/{run_num}/metrics.json"
        with open(json_path) as f:
            rows[(run_dir, RUN_TO_UNIVERSE_SIZE[run_num[-1]])] = json.load(f)

# Construct and visualize a table of 3 panels for the 3 different universe sizes
metrics = pd.DataFrame.from_dict(rows, orient="index")
columns = ["ES80", "Vol", "MD", "SR", "SOR", "ESR", "AAT", "ACT"]
metrics.columns = columns

metrics.index = metrics.index.set_levels(
    metrics.index.levels[1].astype(int), level=1
)
sizes = sorted(metrics.index.get_level_values(1).unique()) 
panels = {size: metrics.xs(size, level=1) for size in sizes}
panels = {size: metrics.xs(size, level=1).T for size in sizes}
for size in sizes:
    panels[size].columns = [c.split('/')[-1] for c in panels[size].columns]

strategy_order = ['equal_weight_optimizer', 'reg_es_asset_full', 'reg_es_asset_zero'] 
panels = {size: panels[size][strategy_order] for size in sizes}

In [38]:
with pd.option_context('display.width', None, 'display.max_columns', None):
    for size in sizes:
        print(f"\n======================================= Universe size: {size} =======================================")
        print(panels[size].round(4))


======================================= Universe size: 10 =======================================
      equal_weight_optimizer  reg_es_asset_full  reg_es_asset_zero
ES80                  1.1540             0.9073             1.1067
Vol                  13.9564            11.0137            13.5930
MD                   54.7697            48.1830            56.6093
SR                    0.1271             0.1459             0.1839
SOR                   0.1739             0.1965             0.2522
ESR                   0.0968             0.1116             0.1423
AAT                   0.1493             0.1482             0.1512
ACT                   0.9182             0.8917             0.0000

======================================= Universe size: 50 =======================================
      equal_weight_optimizer  reg_es_asset_full  reg_es_asset_zero
ES80                  1.0399             0.5674             0.8360
Vol                  12.5001             7.1745            10.317